# CUDA llama-cpp-python Setup

Run this first when using the direct CUDA `llama-cpp-python` path. It installs helper packages, reinstalls the CUDA wheel candidate for `llama-cpp-python`, and checks GPU offload support.


In [ ]:
import importlib.util
import re
import subprocess
import sys

# CUDA wheel setting:
# - 'auto' chooses cu130 when nvidia-smi reports CUDA 13.x, otherwise cu125.
# - For Colab-style CUDA 12.x, cu125 is usually the right wheel.
# - If needed, set explicitly: 'cu124', 'cu125', 'cu130', etc.
LLAMA_CPP_CUDA_WHEEL = 'auto'
REINSTALL_LLAMA_CPP = True

HELPER_PACKAGES = {
    'openpyxl': 'openpyxl',
    'tqdm': 'tqdm',
    'huggingface_hub': 'huggingface_hub',
}


def package_available(import_name):
    return importlib.util.find_spec(import_name) is not None


def pip_install(args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *args])


def detect_cuda_wheel():
    if LLAMA_CPP_CUDA_WHEEL != 'auto':
        return LLAMA_CPP_CUDA_WHEEL
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
        text = (result.stdout or '') + (result.stderr or '')
        match = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', text)
        if match and int(match.group(1)) >= 13:
            return 'cu130'
    except FileNotFoundError:
        pass
    return 'cu125'


print('Notebook Python executable:', sys.executable)
print()
print('GPU status:')
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not found on PATH.')

print()
print('CUDA compiler version, if available:')
try:
    subprocess.run(['nvcc', '--version'], check=False)
except FileNotFoundError:
    print('nvcc not found on PATH. This is okay when using a prebuilt CUDA wheel.')

missing_helpers = [pip_name for import_name, pip_name in HELPER_PACKAGES.items() if not package_available(import_name)]
if missing_helpers:
    print('Installing missing helper package(s):', ', '.join(missing_helpers))
    pip_install(missing_helpers)
else:
    print('Helper packages are already installed.')

cuda_wheel = detect_cuda_wheel()
extra_index_url = f'https://abetlen.github.io/llama-cpp-python/whl/{cuda_wheel}'
print('llama-cpp-python CUDA wheel:', cuda_wheel)
print('Wheel index:', extra_index_url)

if REINSTALL_LLAMA_CPP or not package_available('llama_cpp'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'llama-cpp-python'], check=False)
    pip_install(['--upgrade', '--force-reinstall', '--no-cache-dir', 'llama-cpp-python', '--extra-index-url', extra_index_url])
else:
    print('llama-cpp-python already installed and REINSTALL_LLAMA_CPP is False.')

from llama_cpp import llama_cpp
print('GPU offload support:', llama_cpp.llama_supports_gpu_offload())



In [ ]:
import importlib.util
import subprocess
import sys

print('Notebook Python executable:', sys.executable)
print('llama_cpp import spec found:', importlib.util.find_spec('llama_cpp') is not None)
print()
print('GPU status:')
try:
    subprocess.run(['nvidia-smi'], check=False)
except FileNotFoundError:
    print('nvidia-smi not found on PATH.')
print()
print('Package info:')
subprocess.run([sys.executable, '-m', 'pip', 'show', 'llama-cpp-python'], check=False)



# Prerequisites

## Download Model

In [ ]:
from huggingface_hub import hf_hub_download

# Define the repository and the specific file
repo_id = "unsloth/gemma-4-26B-A4B-it-qat-GGUF"
filename = "gemma-4-26B-A4B-it-qat-UD-Q4_K_XL.gguf"

# Download the file
# This will save it to your local cache and return the path
model_path = hf_hub_download(repo_id=repo_id, filename=filename)

print(f"Model downloaded to: {model_path}")

# Update your GGUF_MODEL_PATH variable with this path
GGUF_MODEL_PATH = model_path

# Occupation Standardization

Notebook for row-wise local GGUF occupation classification using direct CUDA `llama-cpp-python`.


## 0. Variables

Edit this one cell for paths, task instructions, columns, and model settings.

In [ ]:
# Edit this cell first, then run the helper/install/action cells below.

# File path. Output files are generated automatically beside INPUT_PATH.
# Put this notebook and exactly one .csv or .xlsx input file in the same directory.
from pathlib import Path
import os


def find_single_input_file(directory=None):
    directory = Path(directory or Path.cwd())
    candidates = []
    for path in sorted(directory.iterdir()):
        if not path.is_file():
            continue
        if path.name.startswith('~$'):
            continue
        if path.suffix.lower() not in {'.csv', '.xlsx'}:
            continue
        if path.stem.endswith(('_llm', '_parsed', '_classified')):
            continue
        candidates.append(path)
    if len(candidates) == 1:
        return str(candidates[0])
    if not candidates:
        raise FileNotFoundError(f'No .csv or .xlsx input file found in {directory}. Put one beside this notebook or set INPUT_PATH manually.')
    names = '\n'.join(f'- {path.name}' for path in candidates)
    raise ValueError(f'Found multiple .csv/.xlsx input files in {directory}. Keep only one beside this notebook, or set INPUT_PATH manually:\n{names}')


INPUT_PATH = os.getenv('SPREADSHEET_LLM_INPUT_PATH', '../working/data/occupation-standardization/occupation_desc.csv')
SHEET_NAME = 'Sheet1'  # Used only for .xlsx files.
PROCESS_ROW_LIMIT = 50  # Example: 10 for testing, None for all rows. Set None after a successful test run.

# Source text column and output naming.
INPUT_TEXT_COLUMN = 'occupation_desc'
OUTPUT_SUFFIX = '_llm'
OUTPUT_COLUMNS = ['subgroup']

# Plain-language goal plus the generated row prompt instruction.
GOAL = 'Classify each occupation_desc into one standardized occupation subgroup using the provided subgroup taxonomy.'
TASK_INSTRUCTION = "Goal:\nClassify the row's occupation_desc into exactly one standardized occupation subgroup.\n\nOutput schema:\n- Return one compact JSON object with exactly this key: subgroup.\n- The subgroup value must be exactly one of the allowed subgroup names listed below, with identical spelling and punctuation.\n- Use the subgroup names as the only possible labels. Do not output definitions, examples, explanations, occupations, or alternative category names.\n- Use null only when the occupation_desc is blank, unreadable, or not an occupation-like value.\n\nHow to choose:\n- Read only the current row text.\n- Use each subgroup definition and examples as guidance for meaning and boundaries.\n- Prefer the most specific work-function subgroup supported by the text.\n- If the text is vague, generic, incomplete, self-employed without work type, or cannot be reliably mapped to a more specific subgroup, choose Others/Unspecified.\n- Use Proprietor only when the text clearly indicates a business owner/operator and the business activity cannot be classified under another subgroup.\n- Generic entries such as Employee, Worker, Government Employee, Bank Employee, Self-employed, and Others should be Others/Unspecified unless the text includes a clearer work function.\n- Do not infer details that are not supported by the text.\n\nAllowed subgroup names:\n- Proprietor\n- Management\n- Administrative Support\n- Security and Protective Services\n- Transportation and Logistics\n- Customer Service\n- Facilities and Maintenance\n- Culinary Service\n- Agricultural Worker\n- Housekeeping and Domestic Support\n- Architecture and Engineering\n- Skilled Trades\n- Academic Worker\n- Field Operations\n- Accounting and Finance\n- National Security and Public Safety\n- Government Officials and Public Governance\n- Public Service and Program Administration\n- Legal and Judicial\n- Urban Sanitation Worker\n- Healthcare Support\n- Mortuary Professional\n- Media and Broadcasting\n- Clergy\n- Artist\n- Production and Assembly Worker\n- Financial Support Recipient\n- Public Infrastructure Maintenance\n- Market and Street Sales Worker\n- Sales Operations\n- Marketing and Communications\n- Technical / Computer Support Services\n- Occupational Health and Safety Services\n- Sports, Fitness, and Recreation\n- Others/Unspecified\n\nSubgroup reference definitions and examples:\n- Proprietor: Includes individuals who own, operate, and have control over a business or enterprise. Use only when the business activity cannot be clearly classified under another work-function group. Examples: Apartment Rental Owner, Business Owner, Small-scale Proprietor, Store Owner\n- Management: Includes occupations that oversee operations through planning, organizing, leading, controlling, and directing the resources of an organization, business, or department. Examples: Manager, Supervisor\n- Administrative Support: Includes occupations that carry out administrative, clerical, documentation, coordination, and organizational support roles to support business operations. Examples: Office Staff, Data Entry Specialist, Administrative Assistant, Secretary, Executive Assistant, Document Controller\n- Security and Protective Services: Includes occupations that ensure the safety of individuals and properties through guarding, patrolling, and surveillance, usually in private or establishment-based settings. Examples: Security Guard, Watchman, Bouncer\n- Transportation and Logistics: Includes occupations that facilitate the movement of people, goods, documents, freight, cargo, and shipments. Examples: Company Driver, Family Driver, Messenger, Delivery Boy, Delivery Rider, Truck Driver, Pahinante, Taxi Driver, Tricycle Driver, Bus Driver, Sailor, Deckhand\n- Customer Service: Includes occupations that provide frontline assistance to customers, guests, passengers, or clients through inquiries, transactions, hospitality, passenger handling, and direct service support. Examples: Receptionist, Service Crew, Room Attendant, Call Center Employee, Cashier, Tour Guide, Flight Attendant, Airline Ticket Agent\n- Facilities and Maintenance: Includes occupations that maintain the safety, sanitation, cleanliness, and functionality of buildings, offices, and facilities. Examples: Utility and Maintenance Staff, Janitor, Maintenance Staff\n- Culinary Service: Includes occupations that prepare, cook, serve, or manage kitchen and food preparation operations. Examples: Cook, Kitchen Crew, Baker\n- Agricultural Worker: Includes occupations involved in cultivating, planting, harvesting, raising animals, fishing, aquaculture, and producing agricultural or fishery products. Examples: Corn Farmer, Rice Farmer, Vegetables Farmer, Piggery Worker, Poultry Worker, Flockman, Fisherman, Dairy Farmer, Farm Helper\n- Housekeeping and Domestic Support: Includes occupations that routinely maintain the cleanliness, order, and household support needs of homes or residential spaces. Examples: Building and Housekeeping Worker, Household Helper, OFW Domestic Helper\n- Architecture and Engineering: Includes occupations involved in designing, planning, drafting, surveying, engineering, and ensuring the technical or structural integrity of infrastructure, machinery, systems, or technology. Examples: Architect, Civil Engineer, Electrical Engineer, Engineer, Draftsman, Cadet Engineer, Survey Aid\n- Skilled Trades: Includes occupations that perform technical, craft, construction, repair, or maintenance work involving buildings, infrastructure, machinery, equipment, or vehicles. Examples: Construction Worker, Loader Operator, Electrical Technician, Machinery Mechanic, Aircon Repair Worker, Vulcanizing Worker, Electronics Repair Worker\n- Academic Worker: Includes occupations involved in teaching, tutoring, instruction, academic supervision, or formal and non-formal education. Examples: Teacher I, Teacher II, Teacher III, Instructor, ALS Teacher, Online Teacher, Private Teacher, Tutorial/Review Center Teacher, Principal, Online Tutor\n- Field Operations: Includes occupations that perform operational work outside the central office, such as field coordination, inspection, credit, collection, or field support work. Examples: Field Personnel-Credit, Field Personnel-Collection, Field Personnel-Sales, Field Inspector, Field Collector\n- Accounting and Finance: Includes occupations involved in accounting, bookkeeping, auditing, budgeting, treasury, financial advisory, loans, collections analysis, and financial recordkeeping. Examples: Branch Accountant, Accountant, Bookkeeper, Audit Staff, Barangay Treasurer, Loan Officer, Financial Advisor, Account Officer\n- National Security and Public Safety: Includes occupations responsible for protecting the public from threats, enforcing laws, responding to emergencies, or maintaining peace and order. Examples: Sergeant, Airman, Corporal, Soldier, CAFGU, Seaman 1st Class, Local Traffic Enforcer, MMDA Worker, Police Officer, Fireman, Barangay Tanod\n- Government Officials and Public Governance: Includes elected or appointed public officials and governance roles involved in public leadership, legislation, policy implementation, public administration, and community governance. Examples: Mayor, Vice Mayor, Governor, Punong Barangay, Barangay Kagawad, Sangguniang Kabataan Chairman, Sangguniang Panlalawigan Member, Sangguniang Barangay Member\n- Public Service and Program Administration: Includes occupations involved in implementing, coordinating, monitoring, or administering public, community, social welfare, local government, or development programs. Examples: Social Welfare Officer, Local Government Operations Officer, LGU Worker, Development Officer, Development Management Officer, Transportation Development Officer, Program Officer\n- Legal and Judicial: Includes occupations involved in legal practice, legal assistance, dispute resolution, prosecution, judicial services, and upholding laws and regulations. Examples: Prosecutor, Legal Assistant, Lupong Tagapamayapa\n- Urban Sanitation Worker: Includes occupations that maintain the sanitation and cleanliness of public places, streets, and public facilities. Examples: Garbage Collector, Street Sweeper\n- Healthcare Support: Includes occupations that provide patient care, medical assistance, health services, caregiving, or veterinary support. Examples: Dentist Assistant, Medical Worker, Caregiver, Veterinarian, OFW Nurse, Barangay Health Worker\n- Mortuary Professional: Includes occupations related to the care, preparation, and handling of deceased individuals. Examples: Embalmer, Funeral Worker\n- Media and Broadcasting: Includes occupations that produce, present, or disseminate public information through communication platforms such as television, radio, print, or digital media. Examples: Disk Jockey, Reporter, Reporter/Newscaster, News Photographer\n- Clergy: Includes occupations related to leading religious organizations, worship services, and religious rites. Examples: Pastor, Priest\n- Artist: Includes occupations that produce creative works in visual, literary, digital, performance, or entertainment fields. Examples: Painter, Photographer, Graphic Artist, Actor, Singer, Band Member, Tattoo Artist, Recording Artist\n- Production and Assembly Worker: Includes occupations that perform manufacturing, production, assembly, packing, processing, or preparation of goods and products. Examples: Factory Worker, Assembler, Packer, Garments Worker, Handicraft Worker\n- Financial Support Recipient: Includes individuals whose recorded occupation reflects non-employment income or financial support rather than active work. Examples: Pension Recipient, Remittance Recipient\n- Public Infrastructure Maintenance: Includes occupations that monitor, inspect, repair, or maintain public infrastructure, roads, highways, and related public facilities. Examples: Highway Maintenance Worker, Highway/Road Inspector\n- Market and Street Sales Worker: Includes occupations that sell goods in markets, stalls, sidewalks, rolling stores, or informal vending locations. Examples: Frozen Food Vendor, Fish Vendor, Meat Vendor, Rolling Store Vendor, Street Vendor, Market Vendor\n- Sales Operations: Includes occupations that primarily drive revenue through lead generation, promotion, selling, negotiation, account handling, client acquisition, dealership, trading, and closing deals. Examples: Sales Agent, Reservation Sales Agent, Field Sales Representative, Account Executive, Broker, Dealer, Merchandiser, Promodiser, Supplier\n- Marketing and Communications: Includes occupations involved in promotion, advertising, branding, communications, campaigns, events, and marketing support activities. Examples: Marketing Specialist, Marketing Associate, Events Organizer Staff, Brand Marketing Staff\n- Technical / Computer Support Services: Includes occupations that provide technical, computer, IT, system, software, hardware, or digital support services. Examples: IT Support Specialist, System Administrator, Computer Technician\n- Occupational Health and Safety Services: Includes occupations involved in workplace safety, health compliance, risk prevention, and occupational safety monitoring. Examples: Safety Officer, Occupational Health and Safety Staff\n- Sports, Fitness, and Recreation: Includes occupations involved in sports, recreation, coaching, physical activity, and fitness-related services. Examples: Coach, Athlete, Fitness Trainer, Gym Instructor\n- Others/Unspecified: Includes vague, incomplete, unclear, or non-specific occupations that cannot be reliably classified into another subgroup. Examples: Government Employee, Bank Employee, Employee, Worker, Self-employed, Others"

# Local GGUF model settings.
GGUF_MODEL_PATH = os.getenv('GGUF_MODEL_PATH', globals().get('GGUF_MODEL_PATH', ''))  # Example: '/content/models/model.gguf'
N_CTX = 4096  # Fits the full taxonomy prompt; raise only if needed and memory allows.
N_GPU_LAYERS = -1  # If model loading fails, try 0 for CPU-only or a smaller number such as 20.
N_BATCH = 64
N_UBATCH = 64
FLASH_ATTN = False
MAX_TOKENS = 80
VERBOSE_LLAMA = False

print('Variables loaded. Next run Shared Helper Functions, then Run Row-wise LLM Processing.')







## 0.1 Shared Helper Functions

Run once after the variables cell. This loads spreadsheet, JSON parsing, and row processing helpers.

In [ ]:
import csv, json, math, re, time
from pathlib import Path
import os


def pip_install(packages):
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])


try:
    from openpyxl import Workbook, load_workbook
    from tqdm.auto import tqdm
except ImportError:
    pip_install(['openpyxl', 'tqdm'])
    from openpyxl import Workbook, load_workbook
    from tqdm.auto import tqdm


NULL_LIKE_VALUES = {'', 'none', 'null', 'nan', 'n/a', 'na', '-', '--'}
LLM = globals().get('LLM', None)


def normalize_spaces(value):
    return re.sub(r'\s+', ' ', str(value or '')).strip()


def is_blank_like(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    return normalize_spaces(value).lower() in NULL_LIKE_VALUES


def header_map(sheet):
    return {normalize_spaces(cell.value): cell.column for cell in sheet[1] if normalize_spaces(cell.value)}


def require_columns(sheet, names):
    headers = header_map(sheet)
    missing = [name for name in names if name not in headers]
    if missing:
        raise ValueError(f'Missing required exact column name(s): {missing}. Found: {list(headers)}')
    return headers


def last_header_column(sheet):
    return max([cell.column for cell in sheet[1] if normalize_spaces(cell.value)] or [0])


def append_fresh_columns(sheet, names):
    next_col = last_header_column(sheet) + 1
    output_cols = {}
    for name in names:
        sheet.cell(row=1, column=next_col).value = name
        output_cols[name] = next_col
        next_col += 1
    return output_cols


def output_path_for_suffix(input_path, suffix):
    path = Path(input_path)
    if not str(path):
        raise ValueError('Set INPUT_PATH in the Variables cell.')
    if not suffix.startswith('_'):
        raise ValueError('OUTPUT_SUFFIX must start with an underscore, such as _llm, _parsed, or _classified.')
    return path.with_name(f'{path.stem}{suffix}{path.suffix}')


def load_file(path, sheet_name):
    if path.suffix.lower() == '.csv':
        workbook = Workbook()
        sheet = workbook.active
        sheet.title = path.stem[:31] or 'csv'
        with path.open('r', newline='', encoding='utf-8-sig') as handle:
            for row in csv.reader(handle):
                sheet.append(row)
        return workbook, sheet, 'csv'
    if path.suffix.lower() == '.xlsx':
        workbook = load_workbook(path)
        if sheet_name not in workbook.sheetnames:
            raise ValueError(f'Workbook does not contain sheet {sheet_name!r}. Available: {workbook.sheetnames}')
        return workbook, workbook[sheet_name], 'xlsx'
    raise ValueError('INPUT_PATH must end in .xlsx or .csv')


def save_file(workbook, sheet, kind, output_path):
    if kind == 'csv':
        with output_path.open('w', newline='', encoding='utf-8') as handle:
            writer = csv.writer(handle)
            for row in sheet.iter_rows(values_only=True):
                writer.writerow(['' if value is None else value for value in row])
    else:
        workbook.save(output_path)


def load_configured_file():
    path = Path(INPUT_PATH)
    if not str(path) or not path.exists():
        raise ValueError('Set INPUT_PATH to an existing .xlsx or .csv file path in the Variables cell.')
    return load_file(path, SHEET_NAME)


def save_configured_file(workbook, sheet, kind):
    output_path = output_path_for_suffix(INPUT_PATH, OUTPUT_SUFFIX)
    save_file(workbook, sheet, kind, output_path)
    return output_path


def extract_json_object_text(text):
    stripped = str(text or '').strip()
    if stripped.startswith('```'):
        stripped = re.sub(r'^```(?:json)?\s*', '', stripped, flags=re.I)
        stripped = re.sub(r'\s*```$', '', stripped)
    start, end = stripped.find('{'), stripped.rfind('}')
    if start < 0 or end < start:
        raise ValueError('No JSON object found in model response.')
    return stripped[start:end + 1]


def normalize_result(raw):
    if isinstance(raw, str):
        raw = json.loads(extract_json_object_text(raw))
    if not isinstance(raw, dict):
        raise ValueError('Model returned a non-object JSON value.')
    normalized = {}
    for column in OUTPUT_COLUMNS:
        value = raw.get(column)
        if value is None:
            normalized[column] = None
        elif isinstance(value, (dict, list)):
            normalized[column] = json.dumps(value, ensure_ascii=False)
        else:
            text = normalize_spaces(value)
            normalized[column] = None if is_blank_like(text) else text
    return normalized


def build_prompt(row_text):
    columns = ', '.join(OUTPUT_COLUMNS)
    return f'''{TASK_INSTRUCTION}

Return only one valid compact JSON object with exactly these keys:
{columns}

Rules:
- Use null for missing or uncertain values.
- Do not include markdown, explanations, comments, or extra keys.
- Keep values concise and source-grounded.

Row text:
{row_text}
'''


def build_repair_prompt(row_text, previous_output):
    columns = ', '.join(OUTPUT_COLUMNS)
    return f'''The previous response was not valid JSON for the required schema.
Return only one valid compact JSON object with exactly these keys:
{columns}

Use null for missing values. No markdown. No extra keys.

Row text:
{row_text}

Invalid previous response:
{previous_output}
'''


def model_generate(llm, prompt):
    messages = [
        {'role': 'system', 'content': 'You are a strict JSON extraction and classification engine. Return only one compact JSON object.'},
        {'role': 'user', 'content': prompt},
    ]
    output = llm.create_chat_completion(messages=messages, max_tokens=MAX_TOKENS, temperature=0)
    choice = output['choices'][0]
    message = choice.get('message') or {}
    return message.get('content') or choice.get('text') or ''


def process_one_row(llm, row_text):
    first_output = model_generate(llm, build_prompt(row_text))
    try:
        return normalize_result(first_output)
    except Exception:
        return normalize_result(model_generate(llm, build_repair_prompt(row_text, first_output)))


def write_output_row(sheet, row_number, output_columns_by_name, parsed):
    for column_name in OUTPUT_COLUMNS:
        sheet.cell(row=row_number, column=output_columns_by_name[column_name]).value = parsed.get(column_name)


def load_llm_if_needed(llm=None):
    llm = llm or globals().get('LLM')
    if llm is not None:
        return llm
    from llama_cpp import Llama
    model_path = Path(GGUF_MODEL_PATH)
    if not GGUF_MODEL_PATH or not model_path.exists():
        raise ValueError('Set GGUF_MODEL_PATH to an existing .gguf model file path in the Variables cell.')
    print('Loading local model:', model_path)
    print(f'Model settings: N_CTX={N_CTX}, N_GPU_LAYERS={N_GPU_LAYERS}, N_BATCH={N_BATCH}, N_UBATCH={N_UBATCH}, FLASH_ATTN={FLASH_ATTN}')
    try:
        return Llama(
            model_path=str(model_path),
            n_ctx=N_CTX,
            n_gpu_layers=N_GPU_LAYERS,
            n_batch=N_BATCH,
            n_ubatch=N_UBATCH,
            flash_attn=FLASH_ATTN,
            verbose=VERBOSE_LLAMA,
        )
    except ValueError as exc:
        if 'llama_context' in str(exc):
            raise ValueError(
                'Failed to create llama_context. This is usually a memory or model-loading issue. '
                'Try these settings in the Variables cell, then rerun from Shared Helper Functions onward: '
                'N_CTX = 1024, N_BATCH = 64, N_UBATCH = 64, FLASH_ATTN = False. '
                'If it still fails, set N_GPU_LAYERS = 0 for CPU-only loading, or use a smaller/stronger-quantized GGUF model.'
            ) from exc
        raise


def run_row_processing(llm=None):
    llm = load_llm_if_needed(llm)
    workbook, sheet, kind = load_configured_file()
    headers = require_columns(sheet, [INPUT_TEXT_COLUMN])
    input_col = headers[INPUT_TEXT_COLUMN]
    output_columns_by_name = append_fresh_columns(sheet, OUTPUT_COLUMNS)
    last_row = sheet.max_row if PROCESS_ROW_LIMIT is None else min(sheet.max_row, PROCESS_ROW_LIMIT + 1)
    summary = {
        'rows_checked': max(last_row - 1, 0),
        'rows_processed': 0,
        'skipped_blank_input_rows': [],
        'failed_rows': {},
        'appended_columns': OUTPUT_COLUMNS,
    }

    for row_number in tqdm(range(2, last_row + 1), desc='Processing rows with local LLM'):
        row_text = normalize_spaces(sheet.cell(row=row_number, column=input_col).value)
        if not row_text:
            summary['skipped_blank_input_rows'].append(row_number)
            write_output_row(sheet, row_number, output_columns_by_name, {column: None for column in OUTPUT_COLUMNS})
            continue
        try:
            parsed = process_one_row(llm, row_text)
            summary['rows_processed'] += 1
        except Exception as exc:
            summary['failed_rows'][row_number] = str(exc)
            parsed = {column: None for column in OUTPUT_COLUMNS}
        write_output_row(sheet, row_number, output_columns_by_name, parsed)
        time.sleep(0.01)

    saved_path = save_configured_file(workbook, sheet, kind)
    summary['sample_failed_rows'] = dict(list(summary['failed_rows'].items())[:5])
    summary['failed_row_count'] = len(summary.pop('failed_rows'))
    summary['saved_path'] = str(saved_path)
    return summary


print('Shared helper functions loaded for direct CUDA llama-cpp-python backend.')





## 1. Run Row-wise LLM Processing

Loads the local GGUF with `llama-cpp-python`, processes every nonblank row from `INPUT_TEXT_COLUMN`, appends fresh output columns, and saves beside `INPUT_PATH` using `OUTPUT_SUFFIX`.


In [ ]:
LLM = load_llm_if_needed()
summary = run_row_processing(llm=LLM)
print(json.dumps(summary, indent=2))
